# 10. Re-ranking: A Second, Precise Pass Over the Candidates

**RAG Pipeline Series — Notebook 10**

`rag.pdf`'s Chapter 8 opens with a blunt claim: re-ranking is *"one of the highest-ROI optimizations in a RAG pipeline."* Notebooks 7-9 built and measured **first-stage retrievers** (BM25, dense, hybrid) — all designed to be fast over the entire corpus. Speed has a cost: a bi-encoder embeds the query and every document *independently*, so it can never model how specific query words interact with specific document words. A **re-ranker** fixes that in a second pass: it reads a small shortlist of candidates *with* the query, one pair at a time, and reorders them by a much more accurate relevance judgment.

Chapter 8's own framing of the two model types (Table 8.1):

| Property | Bi-Encoder (first-stage) | Cross-Encoder (re-ranker) |
|---|---|---|
| Input | Query and doc encoded separately | Query + doc concatenated |
| Interaction | Only via dot product | Full attention between query and doc |
| Latency | Sub-millisecond | 50-500ms for 100 docs |
| Scalability | Millions/billions of docs | 50-200 docs (requires a first stage) |
| Accuracy | Good (approximate nearest-neighbor) | Excellent (full cross-attention) |

That scalability gap is *why* re-ranking is always a second stage, never a replacement for first-stage retrieval: a cross-encoder scoring every chunk in a large corpus would be far too slow. The standard pipeline is a funnel — retrieve a wide net of candidates cheaply (recall-focused), then re-rank that shortlist precisely (precision-focused).

We use `cross-encoder/ms-marco-MiniLM-L-6-v2` — one of the models Chapter 8 names directly (*"Fast, lightweight, good baseline; trained on MS MARCO passage retrieval"*) — as our re-ranker.

In this notebook we will:
1. Rebuild the dense (bi-encoder) retriever from notebooks 7-9 as the first-stage, retrieving a wider candidate list than we'd normally show a user.
2. Load a cross-encoder re-ranker and score every (query, candidate) pair directly.
3. Watch it re-order a real shortlist — a chunk buried at rank 12 moves to rank 1.
4. Quantify the effect across notebook 9's full eval set with MRR and NDCG@5, before vs. after re-ranking.

## Setup

In [ ]:
%pip install -q -U langchain langchain-community langchain-core sentence-transformers langchain-huggingface langchain-chroma chromadb pandas

## 1. First-stage retrieval: cast a wider net

Re-ranking can only reorder candidates that already made it into the shortlist — it can't recover a chunk the first stage missed entirely (Chapter 8's stage-1/stage-2 funnel again: stage 1 is recall-focused). So we retrieve **more** candidates than we ultimately want (`k=15` here) instead of the `k=5` used in earlier notebooks, giving the re-ranker actual room to work.

In [ ]:
from rag_utils import maybe_colab_upload

# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
maybe_colab_upload()

In [1]:
from rag_utils import build_chroma_store, get_embedder, load_chapter_chunks

pages, full_text, chapters, chunks = load_chapter_chunks()

embeddings = get_embedder()
vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 15})

print(f"{len(chunks)} chunks indexed; first-stage retriever returns up to 15 candidates")

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 55/55 [00:00<00:00, 6262.36it/s]


181 chunks indexed; first-stage retriever returns up to 15 candidates


## 2. Loading a cross-encoder re-ranker

`sentence_transformers.CrossEncoder` loads a model that takes a `(query, document)` pair as a single input — internally formatted as `[CLS] query [SEP] document [SEP]`, per Chapter 8 — and outputs one relevance score per pair. This is a fundamentally different model shape from the bi-encoder in `get_embedder()`: that one encodes a single piece of text into a vector; this one always needs *two* texts together and produces a single number, not a vector.

In [2]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"  # named directly in rag.pdf Chapter 8's model list
reranker = CrossEncoder(RERANKER_MODEL)

print(f"Loaded {RERANKER_MODEL}")

Loading weights: 100%|██████████| 105/105 [00:01<00:00, 103.47it/s]


Loaded cross-encoder/ms-marco-MiniLM-L-6-v2


## 3. Re-ranking a candidate list

Score every `(query, chunk.page_content)` pair from the first-stage shortlist, then sort by that score instead of the bi-encoder's original ranking. `CrossEncoder.predict()` batches this internally, so scoring 15 candidates is one call, not a manual loop of forward passes.

In [3]:
def rerank(query, candidate_docs, top_n=5):
    pairs = [(query, doc.page_content) for doc in candidate_docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidate_docs, scores), key=lambda pair: -pair[1])
    return ranked[:top_n]

## 4. Before vs. after: a chunk buried at rank 12

Take a query from notebook 9's eval set where the dense bi-encoder ranked the correct chunk poorly: *"Give an example of two sentences that mean the same thing but share no words"* — the target is Chapter 4's own paraphrase example (`"The patient presented with chest pain"` / `"The individual experienced angina"`). The bi-encoder buries it at **rank 12** of 15; the specific wording of the query doesn't line up well with the chunk's embedding. Watch what the cross-encoder does once it can read the query and each candidate together.

In [4]:
query = "Give an example of two sentences that mean the same thing but share no words."
target_idx = next(i for i, d in enumerate(chunks) if "The individual experienced angina" in d.page_content)

candidates = dense_retriever.invoke(query)


def mark(doc):
    return " <-- target chunk" if doc.page_content == chunks[target_idx].page_content else ""


print("BEFORE (bi-encoder / dense retrieval order):")
for rank, doc in enumerate(candidates, start=1):
    print(f"  {rank:>2}. chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']}){mark(doc)}")

BEFORE (bi-encoder / dense retrieval order):
   1. chapter=04 (Embeddings)
   2. chapter=06 (Retrieval Techniques)
   3. chapter=04 (Embeddings)
   4. chapter=02 (Evolution of Retrieval)
   5. chapter=09 (Augmentation)
   6. chapter=03 (Data Ingestion)
   7. chapter=02 (Evolution of Retrieval)
   8. chapter=02 (Evolution of Retrieval)
   9. chapter=03 (Data Ingestion)
  10. chapter=04 (Embeddings)
  11. chapter=04 (Embeddings)
  12. chapter=04 (Embeddings) <-- target chunk
  13. chapter=06 (Retrieval Techniques)
  14. chapter=09 (Augmentation)
  15. chapter=03 (Data Ingestion)


In [5]:
reranked = rerank(query, candidates, top_n=len(candidates))

print("AFTER (cross-encoder re-ranked order):")
for rank, (doc, score) in enumerate(reranked, start=1):
    print(f"  {rank:>2}. score={score:.3f}  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']}){mark(doc)}")

AFTER (cross-encoder re-ranked order):
   1. score=-4.202  chapter=04 (Embeddings) <-- target chunk
   2. score=-4.450  chapter=02 (Evolution of Retrieval)
   3. score=-6.345  chapter=02 (Evolution of Retrieval)
   4. score=-7.537  chapter=04 (Embeddings)
   5. score=-7.706  chapter=04 (Embeddings)
   6. score=-7.879  chapter=03 (Data Ingestion)
   7. score=-7.884  chapter=04 (Embeddings)
   8. score=-8.209  chapter=03 (Data Ingestion)
   9. score=-8.545  chapter=09 (Augmentation)
  10. score=-8.888  chapter=02 (Evolution of Retrieval)
  11. score=-9.430  chapter=03 (Data Ingestion)
  12. score=-9.434  chapter=06 (Retrieval Techniques)
  13. score=-9.723  chapter=04 (Embeddings)
  14. score=-10.432  chapter=06 (Retrieval Techniques)
  15. score=-10.622  chapter=09 (Augmentation)


The target chunk jumps from rank 12 to rank 1. The bi-encoder's embedding of the query didn't land close to this chunk's embedding; the cross-encoder, reading the query *against* this specific chunk's text, immediately recognizes it as the paraphrase example being asked about. This is exactly the failure mode Chapter 8 describes: independent encoding can't model fine-grained query-document interaction, and a second pass that reads both together can.

## 5. Quantifying the effect across the full eval set

One anecdote isn't proof. Reusing notebook 9's eval set and metric functions (now in `rag_utils`), compute MRR and NDCG@5 for the dense retriever's raw ranking vs. its re-ranked ordering, across all eight queries.

In [6]:
from rag_utils import build_eval_set, ndcg_at_k, reciprocal_rank

eval_set = build_eval_set(chunks)
content_to_idx = {d.page_content: i for i, d in enumerate(chunks)}

mrr_before, mrr_after, ndcg_before, ndcg_after = [], [], [], []

for item in eval_set:
    candidates = dense_retriever.invoke(item["query"])
    ids_before = [content_to_idx[d.page_content] for d in candidates]

    reranked_docs = [doc for doc, score in rerank(item["query"], candidates, top_n=len(candidates))]
    ids_after = [content_to_idx[d.page_content] for d in reranked_docs]

    relevant = {item["relevant_idx"]}
    mrr_before.append(reciprocal_rank(ids_before, relevant))
    mrr_after.append(reciprocal_rank(ids_after, relevant))
    ndcg_before.append(ndcg_at_k(ids_before, relevant, 5))
    ndcg_after.append(ndcg_at_k(ids_after, relevant, 5))

In [7]:
import pandas as pd

pd.DataFrame({
    "before (dense only)": {
        "mrr": round(sum(mrr_before) / len(mrr_before), 3),
        "ndcg@5": round(sum(ndcg_before) / len(ndcg_before), 3),
    },
    "after (+ re-ranking)": {
        "mrr": round(sum(mrr_after) / len(mrr_after), 3),
        "ndcg@5": round(sum(ndcg_after) / len(ndcg_after), 3),
    },
})

,before (dense only),after (+ re-ranking)
mrr,0.376,0.556
ndcg@5,0.391,0.602


Both metrics improve once the cross-encoder gets a pass over the same 15 candidates — with no change to the underlying chunks, embeddings, or first-stage retriever, purely from re-ordering what was already retrieved.

## 6. What re-ranking can't fix

Re-ranking only *reorders* the first stage's candidate list — it can't add a chunk that never made it into that list. Two of notebook 9's eval queries (the paraphrased hallucination query, and the bi-encoder-limitation query) had their target chunk missing from the dense retriever's top-15 entirely; no amount of re-ranking recovers a candidate the first stage never surfaced. This is exactly why Chapter 8 frames re-ranking as *stage two* of a funnel, not a substitute for stage one — and why notebook 8's hybrid retrieval (better first-stage recall) and notebook 10's re-ranking (better final-stage precision) are complementary, not competing, techniques.

## Takeaways

- A cross-encoder re-ranker reads the query and a candidate document *together*, capturing interactions a bi-encoder's independent embeddings structurally cannot — at the cost of needing one forward pass per candidate, so it only scales to a shortlist (tens to low hundreds), not a whole corpus.
- The standard pipeline is a two-stage funnel: a fast, recall-focused first stage (any of BM25 / dense / hybrid from notebooks 7-8) retrieves a wide candidate list; a slower, precision-focused re-ranker reorders that shortlist.
- Re-ranking is a pure re-ordering step — it cannot rescue a relevant chunk the first stage failed to retrieve at all. First-stage recall and re-ranking precision are two different problems, and a strong pipeline needs both.
- On this notebook's 8-query eval set, adding a re-ranking pass over the same 15 dense-retrieval candidates raised MRR and NDCG@5 with zero changes to chunking, embeddings, or the first-stage retriever itself.

This closes out the retrieval fundamentals covered in this series: loading and chunking (notebooks 1-2), sparse and dense representations (notebooks 3-4), vector storage and metadata filtering (notebooks 5-6), retriever interfaces and hybrid fusion (notebooks 7-8), measurement (notebook 9), and re-ranking (notebook 10) — `rag.pdf`'s own Chapters 9 onward (Augmentation, Generation, Agentic RAG) pick up from here.